<a href="https://colab.research.google.com/github/swadhin94/Python-Test/blob/main/Amazon%20job%20scapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
Amazon Jobs Scraper
Scrapes PM/PPM jobs in India from Amazon Jobs portal.
Requires: selenium, pandas, beautifulsoup4, openpyxl, webdriver-manager
"""

# ── Installs (run once in notebook) ──────────────────────────────────────────
!pip install selenium pandas beautifulsoup4 openpyxl webdriver-manager
!apt-get update -q && wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get install -fy

import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ── Driver Setup ──────────────────────────────────────────────────────────────

def create_driver() -> webdriver.Chrome:
    """Create and return a headless Chrome driver."""
    opts = Options()
    opts.add_argument("--headless")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.binary_location = "/usr/bin/google-chrome"
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=opts
    )


# ── Scraping Helpers ──────────────────────────────────────────────────────────

BASE_URL = (
    "https://www.amazon.jobs/en/search"
    "?offset={offset}"
    "&result_limit=150"
    "&sort=relevant"
    "&category[]=project-program-product-management-non-tech"
    "&country[]=IND"
    "&distanceType=Mi&radius=24km"
    "&industry_experience=seven_plus_years"
    "&base_query=product"
)

def collect_job_links(driver: webdriver.Chrome, max_pages: int = 10) -> list[str]:
    """
    Iterate through paginated search results and collect job detail URLs.

    Args:
        driver:     Active Chrome WebDriver instance.
        max_pages:  Safety cap on the number of pages to visit.

    Returns:
        Deduplicated list of absolute job URLs.
    """
    links: list[str] = []

    for page in range(max_pages):
        offset = page * 10
        url = BASE_URL.format(offset=offset)
        driver.get(url)

        # Wait until at least one job card is present (or time out gracefully)
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "a.job-link"))
            )
        except Exception:
            print(f"  Page {page + 1}: no job links found – stopping pagination.")
            break

        page_links = [
            el.get_attribute("href")
            for el in driver.find_elements(By.CSS_SELECTOR, "a.job-link")
            if el.get_attribute("href")
        ]
        print(f"  Page {page + 1}: found {len(page_links)} job link(s).")
        links.extend(page_links)

        # Stop if the "Next" button is absent or disabled
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, ".pagination-next a")
            if "disabled" in (next_btn.get_attribute("class") or ""):
                break
        except Exception:
            break  # No next button → last page

        time.sleep(1)  # Polite crawl delay

    # Preserve order while deduplicating
    seen: set[str] = set()
    unique: list[str] = []
    for link in links:
        if link not in seen:
            seen.add(link)
            unique.append(link)

    return unique


def scrape_job_details(driver: webdriver.Chrome, url: str) -> tuple[str, str, str]:
    """
    Visit a single job page and extract title, job ID, and full description.

    Args:
        driver: Active Chrome WebDriver instance.
        url:    Full URL of the job detail page.

    Returns:
        (title, job_id, description) – all strings; empty string on failure.
    """
    driver.get(url)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h1.title"))
        )
    except Exception:
        return "", "", ""

    def safe_text(selector: str) -> str:
        try:
            return driver.find_element(By.CSS_SELECTOR, selector).text.strip()
        except Exception:
            return ""

    title   = safe_text("h1.title")
    job_id  = safe_text(".job-id")          # e.g. "Job ID: 2345678" label
    # Fallback: parse job ID from the URL itself
    if not job_id:
        parts = [p for p in url.rstrip("/").split("/") if p.isdigit()]
        job_id = parts[-1] if parts else ""

    description = safe_text(".job-description")

    return title, job_id, description


# ── Classification ────────────────────────────────────────────────────────────

def classify_job(description: str) -> tuple[str, str, str]:
    """
    Tag a job description for key requirements.

    Returns:
        (agile, pm, mba) where each value is 'Y' or 'N'.

    Bug fixed:
        Original code hardcoded pm = 'y' in the else branch so every
        job was marked as requiring PM experience regardless of the text.
    """
    desc = description.lower()

    agile = "Y" if any(kw in desc for kw in ["agile", "scrum", "sprint", "kanban"]) else "N"

    # FIX: was `else "y"` (always Y); now correctly defaults to "N"
    pm    = "Y" if any(kw in desc for kw in ["program management", "project management",
                                               "product management"]) else "N"

    mba   = "Y" if "mba" in desc else "N"

    return agile, pm, mba


# ── Main ──────────────────────────────────────────────────────────────────────

def main(max_pages: int = 5, output_file: str = "amazon_jobs.xlsx") -> pd.DataFrame:
    driver = create_driver()
    records: list[dict] = []

    try:
        print("Step 1 – Collecting job links …")
        job_links = collect_job_links(driver, max_pages=max_pages)
        print(f"Total unique job links collected: {len(job_links)}\n")

        print("Step 2 – Scraping job details …")
        for i, link in enumerate(job_links, start=1):
            print(f"  [{i}/{len(job_links)}] {link}")
            try:
                title, job_id, desc = scrape_job_details(driver, link)
                agile, pm, mba = classify_job(desc)
                records.append({
                    "Title":       title,
                    "Job ID":      job_id,
                    "URL":         link,
                    "Description": desc[:500],   # Truncate for readability
                    "Agile":       agile,
                    "PM":          pm,
                    "MBA":         mba,
                })
            except Exception as exc:
                print(f"    ✗ Failed to scrape {link}: {exc}")
            time.sleep(1)  # Polite crawl delay between detail pages

    finally:
        driver.quit()

    df = pd.DataFrame(records)
    df.to_excel(output_file, index=False)
    df.to_csv(output_file.replace(".xlsx", ".csv"), index=False)
    print(f"\nSaved {len(df)} jobs → {output_file}")
    return df


if __name__ == "__main__":
    df = main(max_pages=11)
    print(df.head())

Hit:1 https://dl.google.com/linux/chrome/deb stable InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,220 B/s)
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 118623 files and directories currently installed.)
Preparing to unp

In [5]:
print(f"Total jobs scraped: {len(df)}")

Total jobs scraped: 10
